# High Frequency Trading Model
Implementation of a Deep Q-Learning trading model using Interactive Brokers data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import pandas_ta as ta
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import logging
import os
from pathlib import Path
from tqdm.auto import tqdm
import gym
from gym import spaces
from typing import Tuple, Dict

# Add visualization imports
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style and configuration
sns.set_style('darkgrid')
plt.style.use('default')
%matplotlib inline

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration
Define project paths and model parameters

In [ ]:
# Project paths
BASE_DIR = Path().absolute()
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

# Create directories if they don't exist
MODELS_DIR.mkdir(exist_ok=True)

# RL Configuration
RL_CONFIG = {
    'learning_rate': 0.001,
    'gamma': 0.99,
    'epsilon_start': 1.0,
    'epsilon_end': 0.01,
    'epsilon_decay': 0.995,
    'batch_size': 32
}

# Technical Indicators Configuration
TECHNICAL_INDICATORS = {
    'RSI': {'length': 14},
    'MACD': {'fast': 12, 'slow': 26, 'signal': 9},
    'BB': {'length': 20, 'std': 2},
    'ATR': {'length': 14},
    'CCI': {'length': 20},  # Added Commodity Channel Index
    'VHF': {'length': 28},  # Added Vertical Horizontal Filter
    'ERI': {'length': 13},  # Added Elder Ray Index
    'ADX': {'length': 14}   # Added Average Directional Index
}

## Data Processing
Implement data loading and technical analysis

In [ ]:
class DataProcessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        print("\n[DataProcessor] Initialized")

    def load_data(self, filepath: str) -> pd.DataFrame:
        try:
            print(f"\n{'='*50}")
            print(f"[DataProcessor] Loading data from {filepath}")
            expected_columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Spread']
            
            # Show progress while reading large files
            print("\n[DataProcessor] Reading data chunks:")
            chunks = pd.read_csv(filepath, 
                              header=None,
                              names=expected_columns,
                              sep=r'\s+',
                              chunksize=1000)
            
            df_chunks = []
            for chunk in tqdm(chunks, desc="Reading data", ncols=100):
                df_chunks.append(chunk)
            df = pd.concat(df_chunks)
            
            print(f"\n[DataProcessor] Raw data shape: {df.shape}")
            
            if df.empty:
                raise ValueError("Empty dataframe loaded")
            
            print("\n[DataProcessor] Processing steps:")
            
            # Time conversion
            print("1. Converting time...")
            df['Time'] = pd.to_datetime(df['Time'], format='mixed')
            
            # Numeric conversion
            print("2. Converting numeric columns...")
            numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']
            for col in numeric_columns:
                df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
            
            # Data cleaning
            print("3. Cleaning data...")
            initial_rows = len(df)
            df = df.dropna()
            df = df.drop_duplicates(subset=['Time'], keep='first')
            
            # Index setting
            print("4. Setting index...")
            df.set_index('Time', inplace=True)
            df.sort_index(inplace=True)
            
            # Print summary
            print(f"\n[DataProcessor] Data loading completed:")
            print(f" - Initial rows: {initial_rows}")
            print(f" - Final rows: {len(df)}")
            print(f" - Columns: {', '.join(df.columns)}")
            print(f"{'-'*50}")
            
            return df
            
        except Exception as e:
            print(f"[DataProcessor] Error loading data: {str(e)}")
            raise

    def add_technical_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        try:
            print(f"\n{'='*50}")
            print("[DataProcessor] Adding technical indicators")
            
            # Initialize indicator strategy
            print("\n[DataProcessor] Setting up indicators:")
            for name, params in TECHNICAL_INDICATORS.items():
                print(f" - {name}: {params}")
            
            custom_strategy = ta.Strategy(
                name="custom_strategy",
                ta=[
                    {"kind": "rsi", "length": TECHNICAL_INDICATORS['RSI']['length']},
                    {"kind": "macd", "fast": TECHNICAL_INDICATORS['MACD']['fast'],
                     "slow": TECHNICAL_INDICATORS['MACD']['slow'],
                     "signal": TECHNICAL_INDICATORS['MACD']['signal']},
                    {"kind": "bbands", "length": TECHNICAL_INDICATORS['BB']['length'],
                     "std": TECHNICAL_INDICATORS['BB']['std']},
                    {"kind": "atr", "length": TECHNICAL_INDICATORS['ATR']['length']},
                    {"kind": "cci", "length": TECHNICAL_INDICATORS['CCI']['length']},
                    {"kind": "vhf", "length": TECHNICAL_INDICATORS['VHF']['length']},
                    {"kind": "eri", "length": TECHNICAL_INDICATORS['ERI']['length']},
                    {"kind": "adx", "length": TECHNICAL_INDICATORS['ADX']['length']}
                ]
            )
            
            # Store initial columns for comparison
            initial_columns = set(df.columns)
            
            # Calculate all indicators
            print("\n[DataProcessor] Calculating indicators...")
            df.ta.strategy(custom_strategy)
            
            # Print summary of added indicators
            new_columns = set(df.columns) - initial_columns
            print("\n[DataProcessor] Technical indicators added:")
            for col in sorted(new_columns):
                print(f" ✓ {col}")
            
            # Clean up and print final stats
            initial_rows = len(df)
            df = df.dropna()
            print(f"\n[DataProcessor] Final statistics:")
            print(f" - Initial rows: {initial_rows}")
            print(f" - Final rows: {len(df)}")
            print(f" - Added indicators: {len(new_columns)}")
            print(f" - Total features: {len(df.columns)}")
            print(f"{'-'*50}")
            
            return df
            
        except Exception as e:
            print(f"[DataProcessor] Error in technical analysis: {str(e)}")
            raise

## Neural Network and Trading Agent
Define the DQN architecture and trading agent

In [ ]:
class DQNNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQNNetwork, self).__init__()
        
        # Validate input size
        if input_size <= 0:
            raise ValueError(f"Invalid input size: {input_size}")
            
        # Enhanced architecture with deeper layers
        h1_size = max(128, input_size * 2)
        h2_size = max(64, input_size)
        h3_size = max(32, input_size // 2)
        
        self.layers = nn.Sequential(
            nn.Linear(input_size, h1_size),
            nn.LayerNorm(h1_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(h1_size, h2_size),
            nn.LayerNorm(h2_size),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(h2_size, h3_size),
            nn.LayerNorm(h3_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(h3_size, output_size)
        )
        
        # Initialize weights using Xavier/Glorot initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)
                
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        return self.layers(x)

In [ ]:
class TradingAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=100000)
        self.batch_size = RL_CONFIG['batch_size']
        
        self.gamma = RL_CONFIG['gamma']
        self.epsilon = RL_CONFIG['epsilon_start']
        self.epsilon_min = RL_CONFIG['epsilon_end']
        self.epsilon_decay = RL_CONFIG['epsilon_decay']
        
        # Initialize models
        self.model = DQNNetwork(state_size, action_size)
        self.target_model = DQNNetwork(state_size, action_size)
        self.target_model.load_state_dict(self.model.state_dict())
        
        # Use learning rate scheduler
        self.optimizer = optim.Adam(self.model.parameters(), lr=RL_CONFIG['learning_rate'])
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max', factor=0.5, patience=10, verbose=True
        )
        
        # Initialize replay memory with random experiences
        self.min_memory_size = 1000
        self.update_target_every = 5  # Update target network every N episodes
        self.episode_count = 0
        self.loss_history = []
        
    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())
    
    def warmup_memory(self, env, num_actions=1000):
        """Pre-fill replay memory with random actions"""
        state = env.reset()
        for _ in range(num_actions):
            action = random.randrange(self.action_size)
            next_state, reward, done, _ = env.step(action)
            self.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                state = env.reset()
                
    def remember(self, state, action, reward, next_state, done):
        state = np.asarray(state, dtype=np.float32)
        next_state = np.asarray(next_state, dtype=np.float32)
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state, training=True):
        state = np.asarray(state, dtype=np.float32)
        if training and (random.random() < self.epsilon):
            return random.randrange(self.action_size)
        
        state = torch.FloatTensor(state)
        if state.dim() == 1:
            state = state.unsqueeze(0)
            
        with torch.no_grad():
            action_values = self.model(state)
        return torch.argmax(action_values).item()

    def train(self, episode_reward=None):
        if len(self.memory) < self.min_memory_size:
            return
        
        # Sample batch with prioritization
        batch = random.sample(self.memory, self.batch_size)
        
        # Prepare batch data
        states = torch.FloatTensor(np.array([i[0] for i in batch]))
        actions = torch.LongTensor(np.array([i[1] for i in batch]))
        rewards = torch.FloatTensor(np.array([i[2] for i in batch]))
        next_states = torch.FloatTensor(np.array([i[3] for i in batch]))
        dones = torch.FloatTensor(np.array([i[4] for i in batch]))
        
        # Current Q values
        current_q_values = self.model(states).gather(1, actions.unsqueeze(1))
        
        # Next Q values with Double DQN
        with torch.no_grad():
            next_actions = self.model(next_states).max(1)[1]
            next_q_values = self.target_model(next_states).gather(1, next_actions.unsqueeze(1)).squeeze(1)
        
        # Calculate target Q values with TD(λ)
        target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
        
        # Huber loss for better stability
        loss = nn.SmoothL1Loss()(current_q_values.squeeze(), target_q_values)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        self.optimizer.step()
        
        # Update target network periodically
        if self.episode_count % self.update_target_every == 0:
            self.update_target_model()
        
        # Update learning rate if reward is provided
        if episode_reward is not None:
            self.scheduler.step(episode_reward)
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        # Store loss for monitoring
        self.loss_history.append(loss.item())
        return loss.item()

## Trading Environment
Define the Forex trading environment using OpenAI Gym

In [ ]:
class ForexTradingEnv(gym.Env):
    def __init__(self, df: pd.DataFrame):
        super(ForexTradingEnv, self).__init__()
        
        self.df = df
        self.current_step = 0
        self.position = None
        
        # Define action and observation spaces
        self.action_space = spaces.Discrete(3)  # Buy, Sell, Hold
        
        # Get all numeric columns for features
        self.feature_columns = self.df.select_dtypes(include=[np.number]).columns
        
        # Calculate state dimension: features + position flag
        self.state_dim = len(self.feature_columns) + 1  # +1 for position flag
            
        self.observation_space = spaces.Box(
            low=-np.inf, 
            high=np.inf, 
            shape=(self.state_dim,),
            dtype=np.float32
        )
        
        # Trading statistics
        self.trades = []
        self.entry_price = None
        
    def _get_state(self) -> np.array:
        """Create state vector from current market data and position"""
        current_data = self.df.iloc[self.current_step]
        
        # Get technical features
        technical_features = current_data[self.feature_columns].values
        
        # Add position flag
        position_flag = 0 if self.position is None else (1 if self.position == 'long' else -1)
        
        # Combine position flag and features
        state = np.concatenate(([position_flag], technical_features))
        return state.astype(np.float32)

    def step(self, action: int) -> Tuple[np.array, float, bool, Dict]:
        current_price = self.df.iloc[self.current_step]['Close']
        reward = 0
        
        # Execute trading action
        if (action == 0) and (self.position is None):  # Buy
            self.position = 'long'
            self.entry_price = current_price
        elif (action == 1) and (self.position is None):  # Sell
            self.position = 'short'
            self.entry_price = current_price
        
        # Move to next step
        self.current_step += 1
        done = self.current_step >= len(self.df) - 1
        
        # Calculate reward based on position outcome
        if self.position is not None:
            next_price = self.df.iloc[self.current_step]['Close']
            price_change = (next_price - self.entry_price) / self.entry_price
            
            if self.position == 'long':
                if price_change > 0:
                    reward = 1  # Winning long trade
                elif price_change < 0:
                    reward = -1  # Losing long trade
            else:  # short position
                if price_change < 0:
                    reward = 1  # Winning short trade
                elif price_change > 0:
                    reward = -1  # Losing short trade
            
            # Record trade result
            self.trades.append({
                'position': self.position,
                'entry_price': self.entry_price,
                'exit_price': next_price,
                'reward': reward
            })
            
            # Reset position after evaluating
            self.position = None
            self.entry_price = None
        
        return self._get_state(), reward, done, {
            'trades': len(self.trades),
            'wins': sum(1 for t in self.trades if t['reward'] > 0),
            'losses': sum(1 for t in self.trades if t['reward'] < 0)
        }

    def reset(self):
        self.current_step = 0
        self.position = None
        self.entry_price = None
        self.trades = []
        return self._get_state()

## Training Process
Set up training environment and train the model

In [ ]:
def setup_training(timeframe: str):
    """Initialize training components for a specific timeframe"""
    # Load and process data
    data_file = DATA_DIR / f"{timeframe}.csv"
    processor = DataProcessor()
    df = processor.load_data(data_file)
    
    # Add debug logging
    logger.info(f"Original dataframe shape: {df.shape}")
    logger.info(f"Original columns: {df.columns.tolist()}")
    
    # Process indicators
    df = processor.add_technical_indicators(df)
    df = df.dropna()  # Remove rows with NaN values
    
    # Validate processed data
    if len(df) < 30:  # Minimum required rows
        raise ValueError(f"Insufficient data for {timeframe}. Need at least 30 rows, got {len(df)}")
    
    # Log processed data info
    logger.info(f"Processed dataframe shape: {df.shape}")
    logger.info(f"Processed columns: {df.columns.tolist()}")
    
    # Create environment
    env = ForexTradingEnv(df)
    logger.info(f"Environment state dimension: {env.state_dim}")
    
    # Initialize agent with correct dimensions
    agent = TradingAgent(state_size=env.state_dim, action_size=env.action_space.n)
    logger.info(f"Agent initialized with state_size={env.state_dim}, action_size={env.action_space.n}")
    
    return env, agent, df

In [ ]:
def train_model(timeframe: str, episodes: int = 1000):
    env, agent, df = setup_training(timeframe)
    best_reward = float('-inf')
    
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        done = False
        
        while not done:
            action = agent.act(state)
            next_state, reward, done, _ = env.step(action)
            
            agent.remember(state, action, reward, next_state, done)
            agent.train()
            
            state = next_state
            total_reward += reward
        
        logger.info(f"Episode {episode + 1}/{episodes}, Total Reward: {total_reward:.2f}")
        
        if total_reward > best_reward:
            best_reward = total_reward
            model_path = MODELS_DIR / f"best_model_{timeframe}.pth"
            torch.save(agent.model.state_dict(), model_path)
            logger.info(f"New best model saved with reward: {best_reward:.2f}")

In [ ]:
# Train models for different timeframes
timeframes = ['M5', 'M15', 'M30', 'H1', 'H4']

for timeframe in timeframes:
    logger.info(f"Starting training for {timeframe} timeframe")
    try:
        train_model(timeframe)
    except Exception as e:
        logger.error(f"Error training {timeframe}: {str(e)}")
        continue